In [1]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from scipy.special import expit

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/processed/toxic_dataset_multilabel.csv"
MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"
OUTPUT_DIR = "../models/toxic-bert-multilabel"

CATEGORIES = [
    "卑語",
    "差別",
    "迷惑行為",
    "猥褻",
    "出会い・プライバシー侵害",
    "違法行為",
    "偏向表現",
]

MAX_LENGTH = 128
THRESHOLD = 0.5
RANDOM_STATE = 42

print("カテゴリ数:", len(CATEGORIES))
print("モデル:", MODEL_NAME)

カテゴリ数: 7
モデル: cl-tohoku/bert-base-japanese-v3


In [3]:
df = pd.read_csv(DATA_PATH)

# 投稿文が空の行を除外
df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str)

# ラベルを0/1の整数型にする
df[CATEGORIES] = df[CATEGORIES].astype(int)

print("データ件数:", len(df))
print("列名:", df.columns.tolist())

display(df[["text"] + CATEGORIES].head())

print("\nカテゴリ別の該当件数")
display(df[CATEGORIES].sum().sort_values(ascending=False))

print(
    "全カテゴリが0の投稿数:",
    (df[CATEGORIES].sum(axis=1) == 0).sum()
)

データ件数: 429
列名: ['id', 'text', 'level_score', 'level', 'classification', 'category', 'annotation_num', '卑語', '差別', '迷惑行為', '猥褻', '出会い・プライバシー侵害', '違法行為', '偏向表現']


,text,卑語,差別,迷惑行為,猥褻,出会い・プライバシー侵害,違法行為,偏向表現
0,いわゆる天使だろ,0,0,0,0,0,0,0
1,底知れないカリスマ性があるよな,0,0,0,0,0,0,0
2,ありがとう,0,0,0,0,0,0,0
3,地味にこれが一番きつい,0,0,0,0,0,0,0
4,ヤクザより怖い,0,1,0,0,0,0,0



カテゴリ別の該当件数


迷惑行為            64
差別              30
偏向表現            17
猥褻              11
卑語               8
違法行為             3
出会い・プライバシー侵害     1
dtype: int64

全カテゴリが0の投稿数: 319


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_STATE,
)

print("学習データ:", len(train_df))
print("検証データ:", len(valid_df))
print("テストデータ:", len(test_df))

print("\n各データのカテゴリ別件数")

for name, data in [
    ("学習", train_df),
    ("検証", valid_df),
    ("テスト", test_df),
]:
    print(f"\n【{name}データ】")
    display(data[CATEGORIES].sum())

学習データ: 343
検証データ: 43
テストデータ: 43

各データのカテゴリ別件数

【学習データ】


卑語               6
差別              24
迷惑行為            54
猥褻               9
出会い・プライバシー侵害     1
違法行為             3
偏向表現            14
dtype: int64


【検証データ】


卑語              2
差別              6
迷惑行為            8
猥褻              2
出会い・プライバシー侵害    0
違法行為            0
偏向表現            2
dtype: int64


【テストデータ】


卑語              0
差別              0
迷惑行為            2
猥褻              0
出会い・プライバシー侵害    0
違法行為            0
偏向表現            1
dtype: int64

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


sample = tokenize_texts(["これは動作確認用の文章です。"])

print("入力文章:", "これは動作確認用の文章です。")
print("トークンID:", sample["input_ids"][0][:20])
print("Attention mask:", sample["attention_mask"][0][:20])

入力文章: これは動作確認用の文章です。
トークンID: [2, 12538, 465, 15763, 13576, 3908, 464, 18530, 13037, 385, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [6]:
def prepare_dataset(dataframe):
    texts = dataframe["text"].tolist()

    labels = dataframe[CATEGORIES].to_numpy(
        dtype=np.float32
    ).tolist()

    encodings = tokenize_texts(texts)
    encodings["labels"] = labels

    return Dataset.from_dict(encodings)


train_dataset = prepare_dataset(train_df)
valid_dataset = prepare_dataset(valid_df)
test_dataset = prepare_dataset(test_df)

print(train_dataset)
print("\n正解ラベルの例:")
print(train_dataset[0]["labels"])
print("ラベル数:", len(train_dataset[0]["labels"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 343
})

正解ラベルの例:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
ラベル数: 7


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    problem_type="multi_label_classification",
)

model.config.id2label = {
    i: category
    for i, category in enumerate(CATEGORIES)
}

model.config.label2id = {
    category: i
    for i, category in enumerate(CATEGORIES)
}

print(model.config.problem_type)
print("出力数:", model.config.num_labels)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8937.72it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING  

multi_label_classification
出力数: 7


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # sigmoidで各カテゴリの確率に変換
    probabilities = expit(logits)

    # 閾値以上を1、それ未満を0とする
    predictions = (
        probabilities >= THRESHOLD
    ).astype(int)

    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "micro_precision": precision_score(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        ),
        "micro_recall": recall_score(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        ),
    }

In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.169359,0.221431,0.000000,0.000000,0.000000,0.000000
2,0.182957,0.207918,0.000000,0.000000,0.000000,0.000000
3,0.084230,0.197175,0.181818,0.057143,1.000000,0.100000
4,0.123806,0.191340,0.260870,0.077922,1.000000,0.150000
5,0.163488,0.190031,0.260870,0.077922,1.000000,0.150000


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.73s/it]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.11it/s]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.61it/s]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-package

In [17]:
test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

print("テスト結果")

for key, value in test_results.items():
    print(f"{key}: {value}")

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Micro F1,Macro F1,Micro Precision,Micro Recall
0.163488,0.061565,5,0.000000,0.000000,0.000000,0.000000


テスト結果
test_loss: 0.06156496703624725
test_micro_f1: 0.0
test_macro_f1: 0.0
test_micro_precision: 0.0
test_micro_recall: 0.0


In [18]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("モデル保存先:", OUTPUT_DIR)

Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.96it/s]

モデル保存先: ../models/toxic-bert-multilabel


In [19]:

from scipy.special import expit
import pandas as pd
import numpy as np

# 検証データで予測
valid_output = trainer.predict(valid_dataset)

valid_probabilities = expit(valid_output.predictions)
valid_true = valid_output.label_ids.astype(int)

# 現在の閾値0.5で0/1に変換
valid_pred = (valid_probabilities >= 0.5).astype(int)

# カテゴリごとに実際の件数と予測件数を集計
category_results = []

for i, category in enumerate(CATEGORIES):
    true_count = int(valid_true[:, i].sum())
    pred_count = int(valid_pred[:, i].sum())

    category_results.append({
        "カテゴリ": category,
        "正解ラベル数": true_count,
        "予測陽性数": pred_count,
        "正解ラベルの最大確率": round(
            float(valid_probabilities[:, i].max()), 4
        ),
        "正解ラベルの平均確率": (
            round(
                float(valid_probabilities[valid_true[:, i] == 1, i].mean()),
                4,
            )
            if true_count > 0 else np.nan
        ),
    })

category_df = pd.DataFrame(category_results)

display(
    category_df.sort_values(
        "正解ラベル数",
        ascending=False,
    )
)


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,カテゴリ,正解ラベル数,予測陽性数,正解ラベルの最大確率,正解ラベルの平均確率
2,迷惑行為,8,3,0.8198,0.3831
1,差別,6,0,0.1838,0.0776
0,卑語,2,0,0.0527,0.0372
3,猥褻,2,0,0.0586,0.0434
6,偏向表現,2,0,0.1541,0.0628
4,出会い・プライバシー侵害,0,0,0.0332,NaN
5,違法行為,0,0,0.0399,NaN


In [22]:

from scipy.special import expit
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# 検証データの予測確率と正解ラベル
valid_output = trainer.predict(valid_dataset)
probabilities = expit(valid_output.predictions)
y_true = valid_output.label_ids.astype(int)

threshold_results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred = (probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Micro Precision": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Micro Recall": recall_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Micro F1": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "予測陽性数": int(y_pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(3).style.format({
        "Threshold": "{:.1f}",
        "Micro Precision": "{:.3f}",
        "Micro Recall": "{:.3f}",
        "Micro F1": "{:.3f}",
        "Macro F1": "{:.3f}",
        "予測陽性数": "{:.0f}"
    })
)


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,Threshold,Micro Precision,Micro Recall,Micro F1,Macro F1,予測陽性数
0,0.1,0.318,0.350,0.333,0.100,22
1,0.2,0.500,0.250,0.333,0.079,10
2,0.3,0.625,0.250,0.357,0.089,8
3,0.4,0.750,0.150,0.250,0.071,4
4,0.5,1.000,0.150,0.261,0.078,3


In [21]:
print("学習データのカテゴリ別陽性件数")
display(train_df[CATEGORIES].sum().sort_values(ascending=False))

print("\n検証データのカテゴリ別陽性件数")
display(valid_df[CATEGORIES].sum().sort_values(ascending=False))

学習データのカテゴリ別陽性件数


迷惑行為            54
差別              24
偏向表現            14
猥褻               9
卑語               6
違法行為             3
出会い・プライバシー侵害     1
dtype: int64


検証データのカテゴリ別陽性件数


迷惑行為            8
差別              6
卑語              2
猥褻              2
偏向表現            2
出会い・プライバシー侵害    0
違法行為            0
dtype: int64

In [23]:

from sklearn.metrics import precision_recall_fscore_support
from scipy.special import expit
import pandas as pd
import numpy as np

# 検証データの予測結果
valid_output = trainer.predict(valid_dataset)
probabilities = expit(valid_output.predictions)
y_true = valid_output.label_ids.astype(int)

# 閾値を0.3に設定
threshold = 0.3
y_pred = (probabilities >= threshold).astype(int)

# カテゴリ別の評価
precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

category_results = pd.DataFrame({
    "カテゴリ": CATEGORIES,
    "正解陽性数": y_true.sum(axis=0),
    "予測陽性数": y_pred.sum(axis=0),
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
})

display(category_results.round(3))


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,カテゴリ,正解陽性数,予測陽性数,Precision,Recall,F1
0,卑語,2,0,0.000,0.000,0.000
1,差別,6,0,0.000,0.000,0.000
2,迷惑行為,8,8,0.625,0.625,0.625
3,猥褻,2,0,0.000,0.000,0.000
4,出会い・プライバシー侵害,0,0,0.000,0.000,0.000
5,違法行為,0,0,0.000,0.000,0.000
6,偏向表現,2,0,0.000,0.000,0.000


In [24]:

import pandas as pd
import numpy as np
from scipy.special import expit

# 検証データの予測
valid_output = trainer.predict(valid_dataset)
probabilities = expit(valid_output.predictions)
y_true = valid_output.label_ids.astype(int)

threshold = 0.3
y_pred = (probabilities >= threshold).astype(int)

# 検証データの文章を取得
texts = valid_df["text"].tolist()

# カテゴリごとに、見逃した文章（FN）を抽出
for i, category in enumerate(CATEGORIES):
    missed_indices = np.where(
        (y_true[:, i] == 1) & (y_pred[:, i] == 0)
    )[0]

    print(f"\n===== {category}：見逃した文章 =====")
    print(f"件数：{len(missed_indices)}")

    for idx in missed_indices[:5]:
        print("-" * 40)
        print("文章：", texts[idx])
        print("正解カテゴリ：", category)
        print(
            "予測確率：",
            round(float(probabilities[idx, i]), 3)
        )


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



===== 卑語：見逃した文章 =====
件数：2
----------------------------------------
文章： 蛇足オナニーを始めるとどんどん本質から遠ざかるだけだ
正解カテゴリ： 卑語
予測確率： 0.04
----------------------------------------
文章： うんシャイは公式じゃなくてゴミ箱視点見ろな
正解カテゴリ： 卑語
予測確率： 0.034

===== 差別：見逃した文章 =====
件数：6
----------------------------------------
文章： このつたない日本語を毎回聞かされるのか
正解カテゴリ： 差別
予測確率： 0.09
----------------------------------------
文章： なんだろみんなお胸がすごいことになってるのに
正解カテゴリ： 差別
予測確率： 0.059
----------------------------------------
文章： 低身長のヒールはやばいって
正解カテゴリ： 差別
予測確率： 0.184
----------------------------------------
文章： ほんと玉無しだな手帳は
正解カテゴリ： 差別
予測確率： 0.033
----------------------------------------
文章： 日本はもうしばらく鎖国しとけ。
正解カテゴリ： 差別
予測確率： 0.061

===== 迷惑行為：見逃した文章 =====
件数：3
----------------------------------------
文章： 伝説の宝見つけた少年みたいな反応やめろ
正解カテゴリ： 迷惑行為
予測確率： 0.184
----------------------------------------
文章： しね
正解カテゴリ： 迷惑行為
予測確率： 0.031
----------------------------------------
文章： もう引退した方が
正解カテゴリ： 迷惑行為
予測確率： 0.038

===== 猥褻：見逃した文章 =====
件数：2
------------------------------------

In [25]:

import pandas as pd

for category in CATEGORIES:
    # 該当カテゴリが正解ラベルとして付いている文章を抽出
    samples = train_df.loc[
        train_df[category] == 1,
        ["text", "category"]
    ].head(10)

    print(f"\n{'=' * 20} {category} {'=' * 20}")
    print(f"表示件数: {len(samples)}")

    for _, row in samples.iterrows():
        print("-" * 50)
        print("文章:", row["text"])
        print("元カテゴリ:", row["category"])



==================== 卑語 ====================
表示件数: 6
--------------------------------------------------
文章: 警官とボディーガードなのにヤンキーは草なんだ
元カテゴリ: 卑語
--------------------------------------------------
文章: ハゲ弱くなったし、上きつくなったか
元カテゴリ: 卑語
--------------------------------------------------
文章: いや糞ラップにその衣装はあわんやろ
元カテゴリ: 卑語
--------------------------------------------------
文章: 本気の孕ませ汁を注入するしかない
元カテゴリ: 卑語|猥褻
--------------------------------------------------
文章: 許可取れてるからって所々舐めくさっとるよな
元カテゴリ: 卑語|迷惑行為
--------------------------------------------------
文章: 撤去作業も映せや
元カテゴリ: 卑語

==================== 差別 ====================
表示件数: 10
--------------------------------------------------
文章: 金持ちが税金対策で買う車でしょ
元カテゴリ: 差別|迷惑行為|偏向表現
--------------------------------------------------
文章: あいつらベトナム人で実験してるから
元カテゴリ: 差別
--------------------------------------------------
文章: 年金天引きの極悪仕様たからな
元カテゴリ: 差別
--------------------------------------------------
文章: 新幹線通ってるんやな
元カテゴリ: 差別
--------------------------------------------------
文章: 二

In [26]:

import pandas as pd

# 学習データのカテゴリ別件数
print("=== カテゴリ別の陽性件数 ===")
display(
    train_df[CATEGORIES]
    .sum()
    .sort_values(ascending=False)
    .rename("陽性件数")
    .to_frame()
)

# 1つの文章に付与されているカテゴリ数
label_counts = train_df[CATEGORIES].sum(axis=1)

print("\n=== 1文章あたりの付与カテゴリ数 ===")
display(
    label_counts.value_counts()
    .sort_index()
    .rename_axis("カテゴリ数")
    .rename("文章数")
    .to_frame()
)

# 2カテゴリ以上が付いた文章の割合
multi_label_ratio = (label_counts >= 2).mean()

print(
    f"\n複数カテゴリが付いた文章の割合: "
    f"{multi_label_ratio:.1%}"
)


=== カテゴリ別の陽性件数 ===


,陽性件数
迷惑行為,54
差別,24
偏向表現,14
猥褻,9
卑語,6
違法行為,3
出会い・プライバシー侵害,1



=== 1文章あたりの付与カテゴリ数 ===


,文章数
カテゴリ数,
0,250
1,79
2,10
3,4



複数カテゴリが付いた文章の割合: 4.1%


In [27]:

# カテゴリ数0の文章を確認
zero_label_df = train_df.loc[
    train_df[CATEGORIES].sum(axis=1) == 0,
    ["text", "category"]
]

print("カテゴリ数0の件数:", len(zero_label_df))

print("\n=== 元のcategory列の内訳 ===")
display(
    zero_label_df["category"]
    .value_counts(dropna=False)
    .rename_axis("元カテゴリ")
    .to_frame("件数")
)

print("\n=== カテゴリ数0の文章サンプル ===")
display(zero_label_df.head(15))


カテゴリ数0の件数: 250

=== 元のcategory列の内訳 ===


,件数
元カテゴリ,
適切,250



=== カテゴリ数0の文章サンプル ===


,text,category
66,それもピンピンしてる,適切
132,すまん、一個上と間違えた。,適切
31,早い方が助かる,適切
84,刃物舐めんな,適切
407,ひ孫に囲まれるとかいいな,適切
168,品川駅停電で大混乱よ,適切
227,餃子王将,適切
415,ダメになるようなら引っ越すわ,適切
399,ツボ押しとマッサージでリラックスを促してる,適切
323,別に,適切


## 重みを入れて再学習
1. カテゴリごとのデータ不均衡に対応する  
「迷惑行為」54件に対し、「違法行為」3件、「出会い・プライバシー侵害」1件と差が大きいです。
2. 損失関数に pos_weight を導入する  
陽性データが少ないカテゴリの誤分類に、より大きなペナルティを与える方法です。
3. 改善前後の評価を比較する  
同じデータ分割で、カテゴリ別のPrecision・Recall・F1を比較します。
 
**まずは pos_weight を導入しよう**  
Hugging Face の Trainer では、モデルの損失計算をカスタマイズすることで実装できます。今の学習コードを大きく変えずに試せます。

In [9]:
import torch
import numpy as np

# 学習データのカテゴリ別陽性・陰性件数
positive_counts = train_df[CATEGORIES].sum().to_numpy(dtype=np.float32)
negative_counts = len(train_df) - positive_counts

# 陽性が少ないカテゴリほど重みを大きくする
pos_weight = torch.tensor(
    negative_counts / np.maximum(positive_counts, 1),
    dtype=torch.float32
)

print("カテゴリ別 pos_weight")
for category, weight in zip(CATEGORIES, pos_weight.tolist()):
    print(f"{category}: {weight:.2f}")


カテゴリ別 pos_weight
卑語: 56.17
差別: 13.29
迷惑行為: 5.35
猥褻: 37.11
出会い・プライバシー侵害: 342.00
違法行為: 113.33
偏向表現: 23.50


In [10]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        # モデルと同じデバイスに重みを移す
        weights = self.pos_weight.to(outputs.logits.device)

        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=weights
        )

        loss = loss_fn(
            outputs.logits,
            labels.float()
        )

        return (loss, outputs) if return_outputs else loss


In [11]:

# 極端な重みを抑える
pos_weight = pos_weight.clamp(max=10.0)

# 以前と同じ設定で新しいモデルを作成
from transformers import AutoModelForSequenceClassification

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    problem_type="multi_label_classification"
)

weighted_model.config.id2label = {
    i: category for i, category in enumerate(CATEGORIES)
}
weighted_model.config.label2id = {
    category: i for i, category in enumerate(CATEGORIES)
}

training_args = TrainingArguments(
    output_dir="../models/toxic-bert-multilabel-wieghted",
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=RANDOM_STATE,
)

weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight
)

weighted_trainer.train()


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 28674.81it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING  

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.699641,0.989503,0.250000,0.047619,0.214286,0.300000
2,0.778817,0.897306,0.322581,0.075188,0.454545,0.250000
3,0.256084,0.970072,0.296296,0.076190,0.571429,0.200000
4,0.450853,0.926450,0.378378,0.131519,0.411765,0.350000
5,0.543832,0.901331,0.333333,0.117293,0.375000,0.300000


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.56s/it]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.69s/it]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.21it/s]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-package

TrainOutput(global_step=430, training_loss=0.5449038075846295, metrics={'train_runtime': 1103.431, 'train_samples_per_second': 1.554, 'train_steps_per_second': 0.39, 'total_flos': 112813929312000.0, 'train_loss': 0.5449038075846295, 'epoch': 5.0})

In [12]:

from sklearn.metrics import precision_recall_fscore_support
from scipy.special import expit
import pandas as pd

# 学習終了後の weighted_trainer を評価
valid_output = weighted_trainer.predict(valid_dataset)

probabilities = expit(valid_output.predictions)
y_true = valid_output.label_ids.astype(int)

# まずは従来と同じ閾値0.5で比較
threshold = 0.5
y_pred = (probabilities >= threshold).astype(int)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

category_results = pd.DataFrame({
    "カテゴリ": CATEGORIES,
    "正解陽性数": y_true.sum(axis=0),
    "予測陽性数": y_pred.sum(axis=0),
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

display(category_results.round(3))


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,カテゴリ,正解陽性数,予測陽性数,Precision,Recall,F1
0,卑語,2,0,0.000,0.000,0.000
1,差別,6,3,0.667,0.333,0.444
2,迷惑行為,8,13,0.385,0.625,0.476
3,猥褻,2,0,0.000,0.000,0.000
4,出会い・プライバシー侵害,0,0,0.000,0.000,0.000
5,違法行為,0,0,0.000,0.000,0.000
6,偏向表現,2,1,0.000,0.000,0.000


In [13]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import precision_recall_fscore_support

# 検証データの予測確率
valid_output = weighted_trainer.predict(valid_dataset)
probabilities = expit(valid_output.predictions)
y_true = valid_output.label_ids.astype(int)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

results = []

for i, category in enumerate(CATEGORIES):
    for threshold in thresholds:
        y_pred = (probabilities[:, i] >= threshold).astype(int)

        precision, recall, f1, _ = (
            precision_recall_fscore_support(
                y_true[:, i],
                y_pred,
                average="binary",
                zero_division=0
            )
        )

        results.append({
            "カテゴリ": category,
            "閾値": threshold,
            "適合率": round(precision, 3),
            "再現率": round(recall, 3),
            "F1": round(f1, 3),
            "予測陽性数": int(y_pred.sum()),
        })

threshold_df = pd.DataFrame(results)
display(threshold_df)

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,カテゴリ,閾値,適合率,再現率,F1,予測陽性数
0,卑語,0.1,0.250,0.500,0.333,4
1,卑語,0.2,1.000,0.500,0.667,1
2,卑語,0.3,0.000,0.000,0.000,0
3,卑語,0.4,0.000,0.000,0.000,0
4,卑語,0.5,0.000,0.000,0.000,0
5,差別,0.1,0.250,0.833,0.385,20
6,差別,0.2,0.231,0.500,0.316,13
7,差別,0.3,0.333,0.333,0.333,6
8,差別,0.4,0.333,0.333,0.333,6
9,差別,0.5,0.667,0.333,0.444,3
